# Python Research Engineering for Systematic Trading

## Classes, Dataclasses, Validation, Testing, and Walk-Forward Research Infrastructure

This notebook focuses on the Python engineering patterns that turn exploratory quantitative research into reusable, configurable, and testable code.

The central example is a small momentum research pipeline. The goal is not to replace vectorized pandas code with object-oriented code everywhere. Instead, the notebook develops judgment about when to use:

- plain variables;
- pure functions;
- `@dataclass` configuration objects;
- reusable classes;
- assertions and explicit exceptions;
- `pytest` tests;
- custom walk-forward research infrastructure.

The final architecture combines a strategy configuration, strategy logic, transaction-cost model, backtester, walk-forward configuration, custom three-way splitter, and tests for invariants such as chronology and no look-ahead.

## Outline

1. Choosing the right abstraction
2. Reproducible synthetic market data
3. Functions first: stateless calculations should stay simple
4. Classes: persistent configuration + related behaviour
5. Dataclasses for result objects
6. Assertions vs explicit exceptions
7. Composition: combine small objects instead of building one giant class
8. Custom three-way walk-forward research infrastructure
9. Testing with plain assert
10. Testing with pytest
11. Numerical tests: pytest.approx, NumPy, and pandas
12. Fixture and parametrization
13. Tests for chronology
14. Test for look-ahead bias

**Final recap**

In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Iterator

import numpy as np
import pandas as pd

TRADING_DAYS = 252
RANDOM_STATE = 42

## 1. Choosing the right abstraction

Use the simplest abstraction that matches the problem.

### Plain variable
Use for one local value.

```python
target_vol = 0.10
```

### Function
Use when the operation is naturally input → output and does not need persistent state.

```python
def annualized_volatility(returns):
    return returns.std() * np.sqrt(252)
```

### Dataclass
Use when several related values describe one configuration or result.

```python
@dataclass(frozen=True)
class StrategyConfig:
    lookback: int
    skip: int
    target_vol: float
```

### Normal class
Use when an object has persistent configuration/state and several related behaviours.

```python
class MomentumStrategy:
    def __init__(self, config):
        self.config = config

    def compute_signal(self, prices):
        ...
```

A class is not automatically more professional than a function. In research code, unnecessary classes make code harder to inspect. The point is to use classes where they reduce repetition and make research infrastructure easier to test.

## 2. Reproducible synthetic market data

We create a small panel of daily prices. The engineering concepts are the focus, so the market process is deliberately simple.

In [2]:
rng = np.random.default_rng(RANDOM_STATE)

dates = pd.bdate_range("2018-01-01", periods=1_800)

asset_names = ["EQ_1", "EQ_2", "FX_1", "CMD_1"]

daily_log_returns = pd.DataFrame(
    rng.normal(
        loc=[0.00025, 0.00020, 0.00005, 0.00015],
        scale=[0.012, 0.011, 0.007, 0.014],
        size=(len(dates), len(asset_names)),
    ),
    index=dates,
    columns=asset_names,
)

prices = 100 * np.exp(daily_log_returns.cumsum())

simple_returns = prices.pct_change()

display(prices.head())

,EQ_1,EQ_2,FX_1,CMD_1
2018-01-01,100.391425,98.882311,100.531725,101.340698
2018-01-02,98.092839,97.495516,100.626760,100.908150
2018-01-03,98.097586,96.604266,101.253170,102.028255
2018-01-04,98.199893,97.829147,101.590150,100.823321
2018-01-05,98.660053,96.822059,102.221879,100.767988


## 3. Functions first: stateless calculations should stay simple

Performance statistics are good examples of pure functions. They do not need object state.

In [3]:
def annualized_volatility(
    returns: pd.Series,
    periods_per_year: int = TRADING_DAYS,
) -> float:
    clean = returns.dropna()

    if clean.empty:
        return np.nan

    return float(clean.std() * np.sqrt(periods_per_year))


def sharpe_ratio(
    returns: pd.Series,
    periods_per_year: int = TRADING_DAYS,
) -> float:
    clean = returns.dropna()

    if clean.empty:
        return np.nan

    ann_vol = annualized_volatility(clean, periods_per_year)

    if ann_vol == 0:
        return np.nan

    return float(clean.mean() * periods_per_year / ann_vol)

In [4]:
example_returns = simple_returns["EQ_1"]

print(f"Annualized volatility: {annualized_volatility(example_returns):.2%}")
print(f"Sharpe ratio:         {sharpe_ratio(example_returns):.2f}")

Annualized volatility: 18.56%
Sharpe ratio:         0.71


## 4. Classes: persistent configuration + related behaviour

A class is useful when several methods operate under the same persistent specification.

The strategy below stores one `StrategyConfig` object and uses it for all signal and weight calculations.

In [5]:
@dataclass(frozen=True)
class StrategyConfig:
    lookback: int = 252
    skip: int = 21
    target_vol: float = 0.10
    vol_lookback: int = 60
    max_leverage: float = 2.0

    def __post_init__(self) -> None:
        if self.lookback <= 0:
            raise ValueError("lookback must be positive")

        if self.skip < 0:
            raise ValueError("skip cannot be negative")

        if self.skip >= self.lookback:
            raise ValueError("skip must be smaller than lookback")

        if self.target_vol <= 0:
            raise ValueError("target_vol must be positive")

        if self.vol_lookback <= 1:
            raise ValueError("vol_lookback must be greater than 1")

        if self.max_leverage <= 0:
            raise ValueError("max_leverage must be positive")

### Why `@dataclass(frozen=True)` here?

`StrategyConfig` mainly stores data. `@dataclass` automatically provides useful boilerplate such as `__init__` and `__repr__`.

`frozen=True` prevents accidental mutation:

```python
config.target_vol = 0.20
```

will fail.

This is useful in research because a configuration should normally remain fixed once an experiment begins.

In [6]:
config = StrategyConfig(
    lookback=252,
    skip=21,
    target_vol=0.10,
    vol_lookback=60,
    max_leverage=2.0,
)

print(config)
print("Lookback:", config.lookback)
print("Target vol:", config.target_vol)

StrategyConfig(lookback=252, skip=21, target_vol=0.1, vol_lookback=60, max_leverage=2.0)
Lookback: 252
Target vol: 0.1


In [7]:
class MomentumStrategy:
    def __init__(self, config: StrategyConfig):
        self.config = config

    def __repr__(self) -> str:
        return (
            f"{self.__class__.__name__}("
            f"lookback={self.config.lookback}, "
            f"skip={self.config.skip}, "
            f"target_vol={self.config.target_vol:.1%})"
        )

    def compute_momentum(self, prices: pd.DataFrame) -> pd.DataFrame:
        return (
            prices.shift(self.config.skip)
            .div(prices.shift(self.config.lookback))
            .sub(1)
        )

    def compute_signal(self, prices: pd.DataFrame) -> pd.DataFrame:
        momentum = self.compute_momentum(prices)

        return np.sign(momentum)

    def compute_raw_weights(self, prices: pd.DataFrame) -> pd.DataFrame:
        signal = self.compute_signal(prices)

        gross = signal.abs().sum(axis=1)

        return signal.div(gross.replace(0, np.nan), axis=0)

    def compute_volatility_scaled_weights(
        self,
        prices: pd.DataFrame,
        returns: pd.DataFrame,
    ) -> pd.DataFrame:
        raw_weights = self.compute_raw_weights(prices)

        raw_strategy_return = (
            raw_weights
            .shift(1)
            .mul(returns)
            .sum(axis=1, min_count=1)
        )

        realized_vol = (
            raw_strategy_return
            .rolling(self.config.vol_lookback)
            .std()
            .mul(np.sqrt(TRADING_DAYS))
        )

        leverage = (
            realized_vol
            .shift(1)
            .rdiv(self.config.target_vol)
            .clip(upper=self.config.max_leverage)
        )

        return raw_weights.mul(leverage, axis=0)

In [8]:
strategy = MomentumStrategy(config)

print(strategy)

signal = strategy.compute_signal(prices)
weights = strategy.compute_volatility_scaled_weights(prices, simple_returns)

display(signal.tail())
display(weights.tail())

MomentumStrategy(lookback=252, skip=21, target_vol=10.0%)


,EQ_1,EQ_2,FX_1,CMD_1
2024-11-18,1.0,1.0,1.0,1.0
2024-11-19,1.0,1.0,1.0,1.0
2024-11-20,1.0,1.0,1.0,1.0
2024-11-21,1.0,1.0,1.0,1.0
2024-11-22,1.0,1.0,1.0,1.0


,EQ_1,EQ_2,FX_1,CMD_1
2024-11-18,0.334226,0.334226,0.334226,0.334226
2024-11-19,0.334227,0.334227,0.334227,0.334227
2024-11-20,0.333796,0.333796,0.333796,0.333796
2024-11-21,0.323641,0.323641,0.323641,0.323641
2024-11-22,0.322471,0.322471,0.322471,0.322471


### Important distinction: attributes vs local variables

Inside:

```python
def compute_signal(self, prices):
```

`prices` is a local method argument.

By contrast:

```python
self.config
```

is persistent object state. Every method can access it.

You generally store something on `self` when it belongs to the identity/configuration/state of the object and needs to persist across method calls.

## 5. Dataclasses for result objects

Dataclasses are also useful for returning several related outputs without using an unstructured tuple or large dictionary.

In [9]:
@dataclass(frozen=True)
class BacktestResult:
    gross_returns: pd.Series
    net_returns: pd.Series
    weights: pd.DataFrame
    turnover: pd.Series
    transaction_costs: pd.Series

    @property
    def gross_equity(self) -> pd.Series:
        return (1 + self.gross_returns.fillna(0)).cumprod()

    @property
    def net_equity(self) -> pd.Series:
        return (1 + self.net_returns.fillna(0)).cumprod()

    @property
    def net_sharpe(self) -> float:
        return sharpe_ratio(self.net_returns)

### Why use `@property`?

A property is accessed like data:

```python
result.net_sharpe
```

but is computed through a method internally.

Use it when the value is naturally an attribute of the object and does not require arguments.

Do not use a property for calculations that need user inputs.

## 6. Assertions vs explicit exceptions

These solve different problems.

### Explicit exception: validate external or configurable inputs

```python
if target_vol <= 0:
    raise ValueError(...)
```

This is appropriate in `StrategyConfig.__post_init__`.

### Assertion: verify an internal invariant during development/testing

```python
assert weights.shape == returns.shape
```

An assertion means: *if the program reached this point, this condition should already be true*.

Do not use `assert` as your primary validation mechanism for user inputs.

In [10]:
def validate_alignment(
    returns: pd.DataFrame,
    weights: pd.DataFrame,
) -> None:
    assert returns.shape == weights.shape
    assert returns.index.equals(weights.index)
    assert returns.columns.equals(weights.columns)

In [11]:
validate_alignment(simple_returns, weights)
print("Alignment checks passed.")

Alignment checks passed.


## 7. Composition: combine small objects instead of building one giant class

The strategy should not also be responsible for modelling transaction costs and orchestrating the complete backtest.

We separate responsibilities and compose objects.

In [12]:
@dataclass(frozen=True)
class CostConfig:
    cost_bps: float = 1.5

    def __post_init__(self) -> None:
        if self.cost_bps < 0:
            raise ValueError("cost_bps cannot be negative")


class TransactionCostModel:
    def __init__(self, config: CostConfig):
        self.config = config

    def compute_turnover(self, weights: pd.DataFrame) -> pd.Series:
        return (
            weights
            .fillna(0)
            .diff()
            .abs()
            .sum(axis=1)
        )

    def compute_costs(self, weights: pd.DataFrame) -> pd.Series:
        turnover = self.compute_turnover(weights)

        return turnover * self.config.cost_bps / 1e4

In [13]:
class Backtester:
    def __init__(
        self,
        strategy: MomentumStrategy,
        cost_model: TransactionCostModel,
    ):
        self.strategy = strategy
        self.cost_model = cost_model

    def run(
        self,
        prices: pd.DataFrame,
        returns: pd.DataFrame,
    ) -> BacktestResult:
        weights = self.strategy.compute_volatility_scaled_weights(
            prices=prices,
            returns=returns,
        )

        validate_alignment(returns, weights)

        gross_returns = (
            weights
            .shift(1)
            .mul(returns)
            .sum(axis=1, min_count=1)
        )

        turnover = self.cost_model.compute_turnover(weights)
        transaction_costs = self.cost_model.compute_costs(weights)

        net_returns = gross_returns - transaction_costs

        assert turnover.dropna().ge(0).all()
        assert transaction_costs.dropna().ge(0).all()

        return BacktestResult(
            gross_returns=gross_returns,
            net_returns=net_returns,
            weights=weights,
            turnover=turnover,
            transaction_costs=transaction_costs,
        )

In [14]:
cost_model = TransactionCostModel(
    CostConfig(cost_bps=1.5)
)

backtester = Backtester(
    strategy=strategy,
    cost_model=cost_model,
)

result = backtester.run(
    prices=prices,
    returns=simple_returns,
)

print(f"Net Sharpe: {result.net_sharpe:.2f}")
display(result.net_equity.tail())

Net Sharpe: -0.14


2024-11-18    0.894442
2024-11-19    0.899055
2024-11-20    0.888524
2024-11-21    0.893016
2024-11-22    0.891983
Freq: B, dtype: float64

### What composition means

`Backtester` **has a** strategy and **has a** cost model.

It does not inherit from either one.

This relationship:

```python
backtester = Backtester(strategy, cost_model)
```

is composition.

For research infrastructure, composition is usually easier to understand and test than deep inheritance hierarchies.

## 8. Custom three-way walk-forward research infrastructure

A custom splitter is a good use case for a dataclass + class.

The configuration stores the research design. The class implements the chronological splitting behaviour.

Each fold is:

```text
TRAIN → VALIDATION → TEST
```

Only the test block is used for recorded performance.

When `expanding=True`, the beginning of the training history stays fixed while the training endpoint moves forward.

In [15]:
@dataclass(frozen=True)
class WalkForwardConfig:
    initial_train_size: int
    validation_size: int
    test_size: int
    step_size: int
    expanding: bool = True

    def __post_init__(self) -> None:
        integer_fields = {
            "initial_train_size": self.initial_train_size,
            "validation_size": self.validation_size,
            "test_size": self.test_size,
            "step_size": self.step_size,
        }

        for name, value in integer_fields.items():
            if value <= 0:
                raise ValueError(f"{name} must be positive")

In [16]:
@dataclass(frozen=True)
class WalkForwardSplit:
    train: np.ndarray
    validation: np.ndarray
    test: np.ndarray

In [17]:
class WalkForwardCV:
    def __init__(self, config: WalkForwardConfig):
        self.config = config

    def split(self, n_samples: int) -> Iterator[WalkForwardSplit]:
        config = self.config

        train_start = 0
        train_end = config.initial_train_size

        while True:
            validation_start = train_end
            validation_end = validation_start + config.validation_size

            test_start = validation_end
            test_end = test_start + config.test_size

            if test_end > n_samples:
                break

            yield WalkForwardSplit(
                train=np.arange(train_start, train_end),
                validation=np.arange(validation_start, validation_end),
                test=np.arange(test_start, test_end),
            )

            train_end += config.step_size

            if not config.expanding:
                train_start += config.step_size

In [18]:
wf_config = WalkForwardConfig(
    initial_train_size=756,
    validation_size=126,
    test_size=63,
    step_size=63,
    expanding=True,
)

cv = WalkForwardCV(wf_config)

splits = list(cv.split(len(prices)))

print(f"Number of folds: {len(splits)}")

for fold_number, split in enumerate(splits[:3], start=1):
    print(
        f"Fold {fold_number}: "
        f"train={split.train[0]}:{split.train[-1]}, "
        f"validation={split.validation[0]}:{split.validation[-1]}, "
        f"test={split.test[0]}:{split.test[-1]}"
    )

Number of folds: 14
Fold 1: train=0:755, validation=756:881, test=882:944
Fold 2: train=0:818, validation=819:944, test=945:1007
Fold 3: train=0:881, validation=882:1007, test=1008:1070


## 9. Testing with plain `assert`

A test is just code that checks expected behaviour.

You do not need `pytest` to understand the underlying principle.

In [19]:
def test_cost_is_zero_when_weights_do_not_change() -> None:
    constant_weights = pd.DataFrame(
        {
            "A": [0.5, 0.5, 0.5],
            "B": [-0.5, -0.5, -0.5],
        }
    )

    model = TransactionCostModel(
        CostConfig(cost_bps=10.0)
    )

    costs = model.compute_costs(constant_weights)

    assert costs.iloc[1:].eq(0).all()


test_cost_is_zero_when_weights_do_not_change()

print("Test passed.")

Test passed.


A useful test should be:

- deterministic;
- small;
- focused on one behaviour;
- easy to interpret when it fails.

Tests should not depend on your full research dataset when a tiny constructed example is enough.

## 10. Testing with `pytest`

`pytest` discovers functions whose names begin with `test_`.

Inside those functions, you normally use ordinary Python `assert`.

Example file:

```python
# tests/test_costs.py

def test_cost_is_zero_when_weights_do_not_change():
    ...
    assert costs.iloc[1:].eq(0).all()
```

Run from a terminal:

```bash
pytest
```

or:

```bash
pytest -q
```

The framework adds discovery, reporting, fixtures, parametrization, and exception-testing utilities around the same basic assertion concept.

### `pytest.raises`

Use it when an input is supposed to raise an exception.

```python
import pytest

def test_invalid_target_vol_raises():
    with pytest.raises(ValueError):
        StrategyConfig(target_vol=-0.10)
```

This verifies that invalid configuration is rejected deliberately.

## 11. Numerical tests: `pytest.approx`, NumPy, and pandas

Floating-point calculations should usually not be compared with exact `==`.

### Scalar

```python
assert result == pytest.approx(expected)
```

### NumPy

```python
np.testing.assert_allclose(actual, expected)
```

### pandas

```python
pd.testing.assert_series_equal(actual, expected)
pd.testing.assert_frame_equal(actual, expected)
```

For quantitative research, the pandas/NumPy assertion helpers are especially useful because they check structure as well as values.

In [20]:
expected = pd.Series(
    [0.01, -0.02, 0.03],
    index=pd.date_range("2026-01-01", periods=3),
)

actual = expected + 1e-12

pd.testing.assert_series_equal(
    actual,
    expected,
    check_exact=False,
    rtol=1e-8,
    atol=1e-10,
)

print("Numerical comparison passed.")

Numerical comparison passed.


## 12. Fixtures and parametrization

Fixtures provide reusable test inputs.

```python
import pytest

@pytest.fixture
def sample_prices():
    ...
    return prices
```

A test can request the fixture by name:

```python
def test_signal_shape(sample_prices):
    signal = strategy.compute_signal(sample_prices)

    assert signal.shape == sample_prices.shape
```

### Parametrization

Use one test definition for several cases:

```python
@pytest.mark.parametrize(
    "cost_bps",
    [0.0, 1.0, 5.0, 10.0],
)
def test_transaction_costs_non_negative(cost_bps):
    ...
```

This is cleaner than copying the same test four times.

## 13. Tests for chronology

A custom financial splitter should be tested aggressively because one indexing mistake can invalidate an entire backtest.

```python
@pytest.fixture
def sample_splits() -> list[WalkForwardSplit]:
    config = WalkForwardConfig(
        initial_train_size=100,
        validation_size=20,
        test_size=10,
        step_size=10,
        expanding=True,
    )
    
    cv = WalkForwardCV(config)

    return list(cv.split(250))

def test_walk_forward_has_strict_chronology(sample_splits: list[WalkForwardSplit]) -> None:
    for split in sample_splits:
        assert split.train[-1] < split.validation[0]
        assert split.validation[-1] < split.test[0]


def test_walk_forward_test_blocks_are_consecutive(sample_splits: list[WalkForwardSplit]) -> None:
    for current, next_ in zip(sample_splits[:-1], sample_splits[1:]):
        assert current.test[-1] + 1 == next_.test[0]


def test_expanding_window_keeps_same_start(sample_splits: list[WalkForwardSplit]) -> None:
    assert all(
        split.train[0] == 0
        for split in sample_splits
    )
```

Would work perfectly here, but pytest.fixture works only if called with pytest. We cannot just call the functions as we do here

In [21]:
def test_walk_forward_has_strict_chronology() -> None:
    config = WalkForwardConfig(
        initial_train_size=100,
        validation_size=20,
        test_size=10,
        step_size=10,
        expanding=True,
    )

    cv = WalkForwardCV(config)

    for split in cv.split(250):
        assert split.train[-1] < split.validation[0]
        assert split.validation[-1] < split.test[0]


def test_walk_forward_test_blocks_are_consecutive() -> None:
    config = WalkForwardConfig(
        initial_train_size=100,
        validation_size=20,
        test_size=10,
        step_size=10,
        expanding=True,
    )

    splits = list(
        WalkForwardCV(config).split(250)
    )

    for current, next in zip(splits[:-1], splits[1:]):
        assert current.test[-1] + 1 == next.test[0]


def test_expanding_window_keeps_same_start() -> None:
    config = WalkForwardConfig(
        initial_train_size=100,
        validation_size=20,
        test_size=10,
        step_size=10,
        expanding=True,
    )

    splits = list(
        WalkForwardCV(config).split(250)
    )

    assert all(
        split.train[0] == 0
        for split in splits
    )

In [22]:
test_walk_forward_has_strict_chronology()
test_walk_forward_test_blocks_are_consecutive()
test_expanding_window_keeps_same_start()

print("Walk-forward tests passed.")

Walk-forward tests passed.


## 14. Test for look-ahead bias

A strong research test is:

> changing future data must not change historical signals.

This directly tests one of the most important invariants in systematic research.

In [23]:
def test_future_prices_do_not_change_past_signal() -> None:
    config = StrategyConfig(
        lookback=60,
        skip=5,
    )

    strategy = MomentumStrategy(config)

    cutoff = prices.index[700]

    original_signal = strategy.compute_signal(prices)

    modified_prices = prices.copy()

    modified_prices.loc[
        modified_prices.index > cutoff
    ] *= 100

    modified_signal = strategy.compute_signal(
        modified_prices
    )

    pd.testing.assert_frame_equal(
        original_signal.loc[:cutoff],
        modified_signal.loc[:cutoff],
    )


test_future_prices_do_not_change_past_signal()

print("No-look-ahead test passed.")

No-look-ahead test passed.


This kind of test is much more valuable in systematic research than a large number of superficial unit tests.

Other useful research invariants include:

- future observations must not affect a fitted historical threshold;
- portfolio weights must respect exposure limits;
- turnover and transaction costs must be non-negative;
- test observations must never enter model selection;
- repeated runs with a fixed random seed should reproduce the same result;
- test engineering such as PCA or scaler should not depend on test but on train.

# Final recap

## Plain functions

Use functions for stateless input → output calculations.

```python
def sharpe_ratio(returns):
    ...
```

## Dataclasses

Use dataclasses for structured configuration or result containers.

```python
@dataclass(frozen=True)
class StrategyConfig:
    lookback: int
    target_vol: float
```

Key syntax:

```python
config.lookback
config.target_vol
```

`__post_init__` validates configuration after initialization.

`frozen=True` prevents accidental mutation.

## Classes

Use a class when related methods share persistent configuration or state.

```python
class Strategy:
    def __init__(self, config):
        self.config = config

    def compute_signal(self, prices):
        return ...
```

- `self` refers to the current object.
- `self.config` persists across method calls.
- ordinary method arguments such as `prices` are temporary inputs.
- `__repr__` controls a useful textual representation.

## Composition

Prefer combining small components:

```python
backtester = Backtester(
    strategy=strategy,
    cost_model=cost_model,
)
```

rather than building one giant class or unnecessary inheritance hierarchy.

## Validation

External/configuration validation:

```python
if target_vol <= 0:
    raise ValueError(...)
```

Internal invariant:

```python
assert weights.shape == returns.shape
```

## Pytest

A test is usually:

```python
def test_something():
    ...
    assert condition
```

Run:

```bash
pytest -q
```

Useful tools:

```python
pytest.raises(...)
pytest.approx(...)
@pytest.fixture
@pytest.mark.parametrize(...)
```

For arrays and labeled data:

```python
np.testing.assert_allclose(...)
pd.testing.assert_series_equal(...)
pd.testing.assert_frame_equal(...)
```